In [1]:
import sys
sys.path.append('../')

In [2]:
import pandas as pd
from sqlalchemy import text
from load_dims import get_engine

In [3]:
engine = get_engine()

Connection successful: (1,)


### Basic Aggregations

Query1: Average electricity prices by state across the enitre date range and all sectors

In [8]:
query1 = """
    SELECT ds.state_short, ROUND(AVG(fp.price_per_kwh),2) as avg_price
    FROM fact_prices fp
    JOIN dim_state ds ON fp.state_id = ds.state_id
    GROUP BY ds.state_short
    ORDER BY avg_price DESC
"""

In [9]:
df = pd.read_sql(text(query1), engine)
df.head(10)

,state_short,avg_price
0,HI,28.04
1,PACN,23.40
2,RI,20.14
3,CA,19.97
4,CT,19.32
5,MA,17.86
6,NEW,17.76
7,AK,16.51
8,PACC,16.06
9,NY,15.32


Query2: Total geneartion for each fuel across all states and years

In [10]:
query2 = """
    select df.fuel_long, SUM(fg.generation ) as total_gen
    from fact_gen fg 
    join dim_fuel df on fg.fuel_id  = df.fuel_id 
    group by df.fuel_long
    order by total_gen desc
"""

In [12]:
df = pd.read_sql(text(query2), engine)
df.head(10)

,fuel_long,total_gen
0,all fuels,20839337.67
1,fossil fuels,12544812.58
2,natural gas & other gases,8623891.78
3,natural gas,8566331.05
4,renewable,4370765.88
5,nuclear,3897800.02
6,all coal products,3830118.34
7,"coal, excluding waste coal",3795316.04
8,all renewables,3091239.37
9,wind,2023462.02


Query 3: average price by state and sector

In [13]:
query3 = """
    SELECT ds.state_short, dp.sector_name, ROUND(AVG(fp.price_per_kwh),2) as avg_price
    FROM fact_prices fp
    JOIN dim_state ds ON fp.state_id = ds.state_id
    join dim_pricesector dp on fp.sector_id =dp.sector_id
    WHERE fp.price_per_kwh  is not NULL
    GROUP BY ds.state_short, dp.sector_name
    ORDER BY avg_price desc
"""

In [14]:
df = pd.read_sql(text(query3), engine)
df.head(10)

,state_short,sector_name,avg_price
0,HI,residential,38.45
1,HI,commercial,35.36
2,HI,all sectors,34.85
3,PACN,residential,32.00
4,HI,industrial,31.55
5,PACN,all sectors,29.21
6,PACN,commercial,28.14
7,PACN,industrial,27.64
8,CA,residential,26.15
9,MA,residential,25.99


Query 4: Top 5 states by NG price

In [15]:
query4 = """
    select ds.state_short , ROUND(avg(ffp.price_per_mmbtu ),2) as avgNG_price
    from fact_fuel_prices ffp
    join dim_state ds on ffp.state_id = ds.state_id 
    join dim_fuel df on ffp.fuel_id = df.fuel_id 
    where df.fuel_short = 'NG'
    group by ds.state_short 
    limit 5
"""

In [16]:
df = pd.read_sql(text(query4), engine)
df.head(10)

,state_short,avgng_price
0,AK,6.52
1,AL,4.44
2,AR,4.12
3,AZ,5.23
4,CA,6.01


Query 5 — States with consistently high prices

In [17]:
query5 = """
    SELECT ds.state_short, ROUND(AVG(fp.price_per_kwh),2) as avg_price
    FROM fact_prices fp
    JOIN dim_state ds ON fp.state_id = ds.state_id
    GROUP BY ds.state_short
    having AVG(fp.price_per_kwh )> 15
    ORDER BY avg_price DESC
"""

In [18]:
df = pd.read_sql(text(query5), engine)
df.head(10)

,state_short,avg_price
0,HI,28.04
1,PACN,23.40
2,RI,20.14
3,CA,19.97
4,CT,19.32
5,MA,17.86
6,NEW,17.76
7,AK,16.51
8,PACC,16.06
9,NY,15.32


Query 6 — Generation mix for California (id = 5), what percentage of total generation comes from each fuel type

In [19]:
query6 = """
    select ds.state_short, df.fuel_long, SUM(fg.generation ) as total_gen, 
	Round((SUM(fg.generation)*100/(select SUM(fg.generation) 
	from fact_gen fg 
	where fg.state_id = 5)),2) as percent_gen
    from fact_gen fg
    join dim_state ds  on fg.state_id = ds.state_id 
    join dim_fuel df  on fg.fuel_id = df.fuel_id 
    where ds.state_short = 'CA'
    group by ds.state_short, df.fuel_long
"""

In [20]:
df = pd.read_sql(text(query6), engine)
df.head(10)

,state_short,fuel_long,total_gen,percent_gen
0,CA,estimated total solar photovoltaic,82182.97,1.89
1,CA,other gases,6909.22,0.16
2,CA,petroleum coke,0.00,0.00
3,CA,natural gas & other gases,473800.29,10.90
4,CA,waste oil and other oils,225.15,0.01
5,CA,renewable,463595.62,10.67
6,CA,conventional hydroelectric,115534.49,2.66
7,CA,solar,193354.51,4.45
8,CA,estimated small scale solar photovoltaic,80294.86,1.85
9,CA,solar thermal,10279.39,0.24


### CTEs

-- Query 7 Average price by state, ranked for states above national average

In [21]:
query7 = """
    with cte_avg_price as (
      	SELECT ds.state_short, ROUND(AVG(fp.price_per_kwh), 2) as avg_price
        FROM fact_prices fp
        JOIN dim_state ds ON fp.state_id = ds.state_id
        GROUP BY ds.state_short
    )
    
    select * 
    from cte_avg_price
    where avg_price > (select avg(fp.price_per_kwh) from fact_prices fp)
    order by avg_price desc
"""

In [22]:
df = pd.read_sql(text(query7), engine)
df.head(10)

,state_short,avg_price
0,HI,28.04
1,PACN,23.40
2,RI,20.14
3,CA,19.97
4,CT,19.32
5,MA,17.86
6,NEW,17.76
7,AK,16.51
8,PACC,16.06
9,NY,15.32


Query 8: Generation mix percentage by state

In [23]:
query8 = """
    with cte_gen_state_fuel as(
    	select ds.state_short, df.fuel_long, SUM(fg.generation ) as fuel_gen 
    	from fact_gen fg
    	join dim_state ds  on fg.state_id = ds.state_id 
    	join dim_fuel df  on fg.fuel_id = df.fuel_id 
    	group by ds.state_short, df.fuel_long
    ),
    cte_total_gen_state as(
    	select ds.state_short, SUM(fg.generation) as state_gen
    	from fact_gen fg 
    	join dim_state ds on fg.state_id = ds.state_id
    	group by ds.state_short 
    )
    
    select gst.state_short, gst.fuel_long, gst.fuel_gen, gs.state_gen,
    	Round((gst.fuel_gen *100/gs.state_gen),2) as percentage_gen
    from cte_gen_state_fuel gst
    join cte_total_gen_state gs on gst.state_short = gs.state_short
    order by gst.state_short
"""

In [24]:
df = pd.read_sql(text(query8), engine)
df.head(10)

,state_short,fuel_long,fuel_gen,state_gen,percentage_gen
0,AK,waste coal,458.65,132238.58,0.35
1,AK,solar,7.81,132238.58,0.01
2,AK,wind,634.73,132238.58,0.48
3,AK,other renewables,0.36,132238.58,0.00
4,AK,waste oil and other oils,1955.50,132238.58,1.48
5,AK,lignite coal,1438.06,132238.58,1.09
6,AK,estimated total solar photovoltaic,49.50,132238.58,0.04
7,AK,all fuels,32974.76,132238.58,24.94
8,AK,landfill gas,199.52,132238.58,0.15
9,AK,municiapl landfill gas,199.52,132238.58,0.15


-- Query 9 Month with peak electricity prices per state

In [25]:
query9 = """
    with cte_state_month as(
    	SELECT ds.state_short as state, fp.date_id as date, ROUND(AVG(fp.price_per_kwh),2) as avg_price
    	FROM fact_prices fp
    	JOIN dim_state ds ON fp.state_id = ds.state_id
    	GROUP BY ds.state_short, fp.date_id
    ),
    cte_max_price as (
    	select csm.state, MAX(csm.avg_price) as max_price
    	from cte_state_month csm
    	group by csm.state
    )
    
    select csm.state, csm.date, cmp.max_price
    from cte_state_month csm
    join cte_max_price cmp
    on csm.state = cmp.state and csm.avg_price = cmp.max_price
    order by csm.state
"""

In [26]:
df = pd.read_sql(text(query9), engine)
df.head(10)

,state,date,max_price
0,AK,202403,18.44
1,AL,202208,10.48
2,AR,202008,13.18
3,AZ,202407,12.53
4,CA,202407,27.21
5,CO,202309,12.89
6,CT,202302,27.49
7,DC,202410,15.93
8,DE,202410,10.68
9,ENC,202402,11.63


Query 10 compare average gas price to electricty price per state

In [27]:
query10 = """
    with cte_avg_price as (
      	SELECT ds.state_short as state, ROUND(AVG(fp.price_per_kwh), 2) as avg_price
        FROM fact_prices fp
        JOIN dim_state ds ON fp.state_id = ds.state_id
        GROUP BY ds.state_short
    ),
    cte_avg_gas as ( 
    	select ds.state_short as state, ROUND(AVG(ffp.price_per_mmbtu),2) as avg_fuel_price
    	from fact_fuel_prices ffp
    	join dim_state ds on ffp.state_id = ds.state_id
    	group by ds.state_short
    )
    
    select ap.state, ap.avg_price, agp.avg_fuel_price
    from cte_avg_price ap 
    join cte_avg_gas agp on ap.state = agp.state
"""

In [28]:
df = pd.read_sql(text(query10), engine)
df.head(10)

,state,avg_price,avg_fuel_price
0,US,11.55,4.33
1,ND,7.02,2.95
2,NV,10.29,5.01
3,OH,9.78,3.38
4,NY,15.32,3.90
5,IN,11.30,3.90
6,WV,8.07,3.62
7,NE,7.23,6.27
8,FL,10.75,5.33
9,ME,13.01,8.40


### Window Functions

-- Query 11 Rank states by average electricity price

In [29]:
query11 = """
    with state_average as (
    	select ds.state_short as state, AVG(fp.price_per_kwh) as average
    	from fact_prices fp
    	join dim_state ds on fp.state_id  = ds.state_id
    	group by ds.state_short
    
    )
    
    select state, average,
    	rank() over (order by average DESC) as price_rank
    from state_average
"""

In [30]:
df = pd.read_sql(text(query11), engine)
df.head(10)

,state,average,price_rank
0,HI,28.041733,1
1,PACN,23.396767,2
2,RI,20.140933,3
3,CA,19.968000,4
4,CT,19.321467,5
5,MA,17.859700,6
6,NEW,17.761100,7
7,AK,16.510700,8
8,PACC,16.058767,9
9,NY,15.318800,10


--Query 12 — Year over year price change by state

In [31]:
query12 = """
    with yearly_state_average as (
    	select ds.state_short as state, dd.year as date_year , Round(AVG(fp.price_per_kwh),2) as average
    	from fact_prices fp
    	join dim_state ds on fp.state_id  = ds.state_id
    	join dim_date dd on fp.date_id = dd.date_id
    	group by ds.state_short, dd.year
    ),
    lag_average as (
    	select state, date_year, average,
    	lag(average,1,0) OVER(partition by state order by date_year) as lag_avg
    	from yearly_state_average ysa
    )
    
    select state, date_year, average, (average - lag_avg) as Yoy
    from lag_average
    order by state, date_year
"""

In [32]:
df = pd.read_sql(text(query12), engine)
df.head(10)

,state,date_year,average,yoy
0,AK,2020,15.60,15.60
1,AK,2021,15.83,0.23
2,AK,2022,16.50,0.67
3,AK,2023,17.01,0.51
4,AK,2024,17.61,0.60
5,AL,2020,7.97,7.97
6,AL,2021,8.25,0.28
7,AL,2022,9.31,1.06
8,AL,2023,9.28,-0.03
9,AL,2024,9.61,0.33


-- Query 13 — Running total of generation by state california and natural gas

In [33]:
query13 = """
    with total_gen as (
    	select fg.date_id , SUM(fg.generation )  as total_gen
    	from fact_gen fg
    	join dim_state ds on ds.state_id = fg.state_id
    	join dim_fuel df on df.fuel_id = fg.fuel_id
    	where ds.state_short = 'CA' and df.fuel_short = 'NG'
    	group by fg.date_id
    )
    
    select date_id, total_gen,
    	SUM(total_gen) over (order by date_id) as running_gen
    from total_gen as tg
"""

In [34]:
df = pd.read_sql(text(query13), engine)
df.head(10)

,date_id,total_gen,running_gen
0,202001,6897.68,6897.68
1,202002,6083.05,12980.73
2,202003,6764.08,19744.81
3,202004,4768.63,24513.44
4,202005,4488.33,29001.77
5,202006,6287.20,35288.97
6,202007,8873.23,44162.20
7,202008,11305.75,55467.95
8,202009,9683.13,65151.08
9,202010,10137.40,75288.48


-- Query 14 Rank fuel types within each state by total generation select top ranked fuel

In [35]:
query14 = """
    with state_fuel_gen as(
    	select ds.state_short, df.fuel_short , SUM(fg.generation )  as total_gen
    	from fact_gen fg
    	join dim_state ds on ds.state_id = fg.state_id
    	join dim_fuel df on df.fuel_id = fg.fuel_id
    	where not fuel_short = 'ALL'
    	group by ds.state_short, df.fuel_short
    
    )
    
    select *
    from (
    	select *, rank () over (partition by stg.state_short order by stg. total_gen DESC) as ranked
    	from state_fuel_gen stg
    ) as ranked_fuel
    where ranked_fuel.ranked = 1
"""

In [36]:
df = pd.read_sql(text(query14), engine)
df.head(10)

,state_short,fuel_short,total_gen,ranked
0,AK,FOS,23496.81,1
1,AL,FOS,413141.77,1
2,AR,FOS,206155.44,1
3,AZ,FOS,315349.36,1
4,CA,FOS,475498.12,1
5,CO,FOS,183378.39,1
6,CT,FOS,124306.92,1
7,DC,TPV,560.59,1
8,DC,TSN,560.59,1
9,DE,FOS,23540.20,1
